# COVER-KBC Profile E3 - Colab execution

Thin driver for the AKBC Shared Task 2026 system. Every prediction path calls repository code; no COVER logic is reimplemented here.

**Current best profile**

| item | value |
|---|---|
| profile | Profile E3 - Mistral Area Multi-View |
| config | `configs/experiments/cover_kbc_v3_7_profile_e3_mistral_area_multiview_test.yaml` |
| model | `mistralai/Mistral-Small-3.2-24B-Instruct-2506` |
| revision | `95a6d26c4bfb886c58daf9d3f7332c857cb27b43` |
| unique parameters | 24,011,361,280 / 32,000,000,000 |
| hidden TEST macro-F1 | 0.5857 |

Profile E3 uses one physical Mistral runtime for every logical role. Qwen is not active in this notebook or in the current best config.

**Runtime:** set Runtime -> Change runtime type -> GPU before running neural inference.

## 1. Clone the repository

In [ ]:
REPO_URL = "https://github.com/YOUR_ORG/FactElicit-AKBC.git"  # set for your fork
COMMIT = ""  # optional exact commit

import os, subprocess

if not os.path.isdir("FactElicit-AKBC"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir("/content/FactElicit-AKBC")
if COMMIT:
    subprocess.run(["git", "checkout", COMMIT], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip())

## 2. Install dependencies

In [ ]:
!pip -q install -e '.[hf]'
!pip -q install bitsandbytes
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu", p.name, round(p.total_memory / 2**30, 1), "GiB")
print("transformers", transformers.__version__)

## 3. Optional auth and persistent cache
Mistral-Small may require accepting the model license on Hugging Face. Mount Drive only to persist cache and artifacts; do not add any external factual source to inference.

In [ ]:
USE_HF_TOKEN = False
USE_DRIVE = False

if USE_HF_TOKEN:
    from huggingface_hub import login
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["HF_HOME"] = "/content/drive/MyDrive/cover_kbc/hf-cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print("HF_HOME =", os.environ["HF_HOME"])

## 4. Verify repository and budget

In [ ]:
CONFIG = "configs/experiments/cover_kbc_v3_7_profile_e3_mistral_area_multiview_test.yaml"

dirty = subprocess.run(["git", "status", "--porcelain", "benchmark/"], capture_output=True, text=True).stdout.strip()
assert not dirty, f"benchmark/ has been modified:\n{dirty}"
print("benchmark/ is clean")

!python scripts/audit_model_budget.py {CONFIG}

## 5. Optional zero-model dry run

In [ ]:
!python scripts/run_area_multiview.py --config {CONFIG} --split test --output-dir outputs/e3_area_multiview_dry_run --dry-run

## 6. Run Profile E3
Use `split = val` for local scoring and small `limit` values for smoke runs. Use `split = test`, `limit = 0`, and `--no-eval` for blind TEST prediction generation.

In [ ]:
SPLIT = "val"   # "train" | "val" | "test"
LIMIT = 20      # 0 = full split
OUT_DIR = ""    # optional explicit output directory

cmd = ["python", "scripts/run_cover.py", "--config", CONFIG, "--split", SPLIT]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if OUT_DIR:
    cmd += ["--output-dir", OUT_DIR]
if SPLIT == "test":
    cmd += ["--no-eval"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 7. Inspect latest artifacts

In [ ]:
import json, pathlib

runs = sorted(pathlib.Path("outputs").glob("*"), key=lambda p: p.stat().st_mtime)
assert runs, "no output directory found"
run = pathlib.Path(OUT_DIR) if OUT_DIR else runs[-1]
print("RUN_DIR =", run)
for name in sorted(p.name for p in run.iterdir()):
    print(f"  {name:36s} {(run / name).stat().st_size:>12,} bytes")

manifest = run / "manifest.json"
if manifest.is_file():
    print(json.dumps(json.loads(manifest.read_text()), indent=2)[:5000])

## 8. Package a submission

In [ ]:
# Only package after reviewing RUN_DIR and confirming this is the intended artifact.
# !python scripts/package_submission.py --predictions {run}/predictions.jsonl --output outputs/submission.zip

---

Closed-book rule: do not add web search, RAG, external KB lookup, or manually authored subject-answer tables to the prediction path. Model downloads are environment setup, not factual retrieval.